<a href="https://colab.research.google.com/github/sophiannunez18/DSC_110/blob/main/ML_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [74]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [75]:
df = pd.read_csv("https://raw.githubusercontent.com/osoliman/DSC110/refs/heads/main/Datasets/nhanes_final.csv")

In [76]:
df.head()

,id,gender,age,race_ethnicity,education,bmi,height_cm,systolic_bp,has_diabetes,hba1c,fasting_glucose
0,130378.0,1.0,43.0,6.0,5.0,27.0,179.5,135.0,2.0,5.6,113.0
1,130379.0,1.0,66.0,3.0,5.0,33.5,174.2,121.0,2.0,5.6,99.0
2,130380.0,2.0,44.0,2.0,3.0,29.7,152.9,111.0,1.0,6.2,156.0
3,130381.0,2.0,5.0,7.0,NaN,23.8,120.1,NaN,2.0,NaN,NaN
4,130382.0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN


Step 1: Filtering

In [77]:
df = df[df['age'] >= 18]

In [78]:
df['age'].min()

18.0

Step 2: Drop columns/fields

Drop irrelevant fields eg., id and fields that are not avaliable at the time of prediction e.g., hba1c

In [79]:
df = df.drop(columns=['id', 'hba1c', 'fasting_glucose'])

In [80]:
df.head()

,gender,age,race_ethnicity,education,bmi,height_cm,systolic_bp,has_diabetes
0,1.0,43.0,6.0,5.0,27.0,179.5,135.0,2.0
1,1.0,66.0,3.0,5.0,33.5,174.2,121.0,2.0
2,2.0,44.0,2.0,3.0,29.7,152.9,111.0,1.0
6,1.0,43.0,1.0,2.0,NaN,NaN,NaN,2.0
7,2.0,65.0,3.0,3.0,NaN,NaN,NaN,2.0


Step 3: Mapping Labels
Gender, Education, Race

In [81]:
#creating dictionaries and using map function
gender_map = {1.0: 'Male', 2.0: 'Female'}
df['gender'] = df['gender'].map(gender_map)

In [82]:
df['gender'].value_counts()

,count
gender,
Female,4496
Male,3657


In [83]:
#Education
education_map = {
    1.0: 'Less than 9th',
    2.0: '9th - 11th grade',
    3.0: 'High School',
    4.0: 'Some college',
    5.0: 'College grad',
    9.0: 'Do not know'

}

In [84]:
df['education'] = df['education'].map(education_map)

In [85]:
df['education'].value_counts()

,count
education,
College grad,2625
Some college,2370
High School,1749
9th - 11th grade,666
Less than 9th,373
Do not know,11


In [86]:
#Race_Ethnicity mapping
race_map = {
    1.0: 'Mexican American',
    2.0: 'Other Hispanic',
    3.0: 'White',
    4.0: 'Black',
    6.0: 'Asian',
    7.0: 'Other'
}
df['race_ethnicity'] = df['race_ethnicity'].map(race_map)
df['race_ethnicity'].value_counts()

,count
race_ethnicity,
White,4704
Black,1045
Other Hispanic,819
Mexican American,598
Other,544
Asian,443


Step 4: Missing values

In [87]:
#find fields with missing values
df.isna().sum()

,0
gender,0
age,0
race_ethnicity,0
education,359
bmi,1918
height_cm,1891
systolic_bp,2030
has_diabetes,3


In [88]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

In [89]:
#loops
for col in ['bmi', 'height_cm', 'systolic_bp'] :
  df[col] = df[col].fillna(df[col].mean())

In [90]:
df.isna().sum()

,0
gender,0
age,0
race_ethnicity,0
education,359
bmi,0
height_cm,0
systolic_bp,0
has_diabetes,3


In [91]:
#removing rows where education is missing
df = df.dropna(subset=['education'])

In [92]:
df.isna().sum()

,0
gender,0
age,0
race_ethnicity,0
education,0
bmi,0
height_cm,0
systolic_bp,0
has_diabetes,0


Step 5: Define the target

In [93]:
#Include only rows that have diabetes yes or no
#Removing rows where diabetes is borderline or missing

df = df[df['has_diabetes'].isin([1.0,2.0])]

In [94]:
df['has_diabetes'].value_counts()

,count
has_diabetes,
2.0,6455
1.0,1068


In [95]:
df['has_diabetes'] = df['has_diabetes'].map({
    1.0: 1,
    2.0: 0
})

In [98]:
df['has_diabetes'].value_counts()

,count
has_diabetes,
0,6455
1,1068


Step 6: One-hot encoding

In [100]:
df = pd.get_dummies(df,
                    columns = ['gender', 'race_ethnicity', 'education'],
                    drop_first = True)

In [101]:
df.head()

,age,bmi,height_cm,systolic_bp,has_diabetes,gender_Male,race_ethnicity_Black,race_ethnicity_Mexican American,race_ethnicity_Other,race_ethnicity_Other Hispanic,race_ethnicity_White,education_College grad,education_Do not know,education_High School,education_Less than 9th,education_Some college
0,43.0,27.000000,179.500000,135.000000,0,True,False,False,False,False,False,True,False,False,False,False
1,66.0,33.500000,174.200000,121.000000,0,True,False,False,False,False,True,True,False,False,False,False
2,44.0,29.700000,152.900000,111.000000,1,False,False,False,False,True,False,False,False,True,False,False
6,43.0,29.659326,166.954982,122.561653,0,True,False,True,False,False,False,False,False,False,False,False
7,65.0,29.659326,166.954982,122.561653,0,False,False,False,False,False,True,False,False,True,False,False


Step 7: Scaling numeric fields

Not required for all ML algorithms, but mostly required for neural network algorithms

In [105]:
numeric_cols = ['age', 'bmi', 'height_cm', 'systolic_bp']
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [106]:
df.head()

,age,bmi,height_cm,systolic_bp,has_diabetes,gender_Male,race_ethnicity_Black,race_ethnicity_Mexican American,race_ethnicity_Other,race_ethnicity_Other Hispanic,race_ethnicity_White,education_College grad,education_Do not know,education_High School,education_Less than 9th,education_Some college
0,-0.589355,-0.425106,1.424574,0.758296,0,True,False,False,False,False,False,True,False,False,False,False
1,0.719517,0.589889,0.823127,-0.115670,0,True,False,False,False,False,True,True,False,False,False,False
2,-0.532448,-0.003493,-1.594009,-0.739931,1,False,False,False,False,True,False,False,False,True,False,False
6,-0.589355,-0.009844,0.000959,-0.018182,0,True,False,True,False,False,False,False,False,False,False,False
7,0.662610,-0.009844,0.000959,-0.018182,0,False,False,False,False,False,True,False,False,True,False,False


In [108]:
df[numeric_cols].describe().round(2)

,age,bmi,height_cm,systolic_bp
count,7523.00,7523.00,7523.00,7523.00
mean,0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00
min,-1.90,-2.91,-3.85,-2.99
25%,-0.87,-0.61,-0.61,-0.55
50%,0.15,-0.01,0.00,-0.02
75%,0.83,0.29,0.53,0.32
max,1.52,7.04,3.83,6.81


Train test split

In [109]:
X = df.drop(columns=['has_diabetes'])
y = df['has_diabetes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [110]:
print(X_train.shape, y_train.shape)

(6018, 15) (6018,)


In [111]:
print(X_test.shape, y_test.shape)

(1505, 15) (1505,)
